# 1.0 — ATC Catalog and Scraper

Explores the ATC/DDD catalog (`src/data/atc/codes.json`) and the scraper's pure
functions (`scripts/atc_scraper.py`) **without network access**. The final
section makes real requests and is optional.

Notebook = laboratory: logic that matures here graduates to a module in
`src/` with tests.

## Import Bootstrap

An `.ipynb` is not part of a package, so a relative import (`from ..src`)
fails. We add the repo root (and `scripts/`) to `sys.path` so `from src...
import` statements work.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent  # the notebook lives in notebooks/ -> repo root
for p in (ROOT, ROOT / "scripts"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

%load_ext autoreload
%autoreload 2  # changes in src/ reload without restarting the kernel
print("ROOT =", ROOT)

## Catalog: size and distribution by level

Code length → ATC hierarchical level: `1->1, 3->2, 4->3, 5->4, 7->5`.

In [ ]:
import json
from collections import Counter

codes = json.loads((ROOT / "src/data/atc/codes.json").read_text())
print("total codes:", len(codes))

LEVEL_BY_LEN = {1: 1, 3: 2, 4: 3, 5: 4, 7: 5}
by_level = Counter(LEVEL_BY_LEN.get(len(c), 0) for c in codes)
for level in sorted(by_level):
    print(f"  level {level}: {by_level[level]:>5} codes")

In [ ]:
# Show a couple of codes from each level
samples = {}
for code, name in codes.items():
    lvl = LEVEL_BY_LEN.get(len(code), 0)
    samples.setdefault(lvl, []).append((code, name))

for level in sorted(samples):
    print(f"level {level}:")
    for code, name in samples[level][:2]:
        print(f"  {code:<8} {name}")

## `ATCCode` on real data

The model resolves the name from `codes.json` and derives level / parents /
pharmacological class. `ATCCode` reads the JSON via `Path(__file__)`, so it
does not depend on the cwd.

In [ ]:
from src.data.schemas import ATCCode

for code in ("J01CA04", "C09AA05", "A10BA02"):  # amoxicillin, ramipril, metformin
    atc = ATCCode(code=code)
    print(f"{atc.code}")
    print(f"  name                 : {atc.name}")
    print(f"  level                : {atc.level}")
    print(f"  parent (level 3)     : {atc.get_parent_code(level=3)}")
    print(f"  pharmacological_class: {atc.pharmacological_class}")
    print()

## Scraper: pure functions (offline)

`extract_code_from_href` and `parse_links` never touch the network: they are
tested against a sample HTML fragment that mimics the site's `?code=...`
links.

In [ ]:
from atc_scraper import extract_code_from_href, parse_links

sample_html = '''
<html><body>
  <a href="./?code=A10&showdescription=no">A10</a>
  <a href="./?code=A10B&showdescription=no">A10B</a>
  <a href="./?code=A10BA&showdescription=no">Biguanides</a>
  <a href="https://example.org/other">link without code</a>
</body></html>
'''

print("extract_code_from_href:")
print("  ", extract_code_from_href("./?code=A10BA&showdescription=no"))  # -> 'A10BA'
print("  ", extract_code_from_href("https://example.org/other"))         # -> None

pairs = parse_links(sample_html)
print("\nparse_links ->")
for code, name in pairs:
    print(f"  {code:<6} {name}")

assert ("A10BA", "Biguanides") in pairs

## Scraper against the network — OPTIONAL ⚠️

The cells below make **real requests** to `atcddd.fhi.no` (with `time.sleep`
between them). They are not required to validate the notebook. They walk only
a small subtree (`A10`, antidiabetics) to avoid hammering the site.

In [ ]:
# from atc_scraper import scrape_recursive
# import tempfile
#
# results, visited = {}, set()
# checkpoint = Path(tempfile.mkstemp(suffix=".json")[1])
# scrape_recursive("A10", results, visited, checkpoint)  # download the A10 subtree
# print(len(results), "codes under A10")
# {c: results[c] for c in sorted(results)[:10]}